In [1]:
REPO_URL = "https://github.com/EfficientNN/Efficient-Models-course-ITMO-2026.git"
REPO_DIR = "/content/Efficient-Models-course-ITMO-2026"

import os, subprocess, sys

if not os.path.isdir(REPO_DIR):
    try:
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR])
    except subprocess.CalledProcessError:
        os.makedirs(os.path.join(REPO_DIR, "src"), exist_ok=True)
        print("Clone не удался. Загрузите файлы из src/ в", os.path.join(REPO_DIR, "src"))
else:
    print("Репозиторий уже есть:", REPO_DIR)

os.chdir(REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
print("cwd:", os.getcwd())
print("src files:", sorted(os.listdir("src")) if os.path.isdir("src") else "НЕТ src/")

Репозиторий уже есть: /content/Efficient-Models-course-ITMO-2026
cwd: /content/Efficient-Models-course-ITMO-2026
src files: ['__init__.py', '__pycache__', 'bench_llama.py', 'benchmark.py', 'linear.py', 'llama_shapes.py', 'matmul.py', 'quantize.py', 'quantize_llama.py', 'tests.py', 'utils.py']


In [2]:
import importlib.util, subprocess, sys

def ensure(pkg, import_name=None):
    name = import_name or pkg
    if importlib.util.find_spec(name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

ensure("triton")
ensure("transformers")
ensure("accelerate")
ensure("huggingface_hub")

import torch, triton
assert torch.cuda.is_available(), "Включите GPU: Runtime → Change runtime type → GPU"
major, minor = torch.cuda.get_device_capability()
print(f"torch {torch.__version__} | triton {triton.__version__} | CUDA {torch.version.cuda}")
print(f"GPU: {torch.cuda.get_device_name(0)} (sm_{major}{minor})")
if major < 8:
    print("bf16 tensor cores нет — матмул пойдёт через fp32 (корректно, но медленнее).")

torch 2.11.0+cu130 | triton 3.6.0 | CUDA 13.0
GPU: Tesla T4 (sm_75)
bf16 tensor cores нет — матмул пойдёт через fp32 (корректно, но медленнее).


## Тесты ядер

In [3]:
from tests import run_all_tests
run_all_tests()

[OK  ] Эталон: pack -> unpack возвращает исходные коды: 8/8
        например: 1x1
[OK  ] Эталон: формат упаковки на известном векторе: 1/1
        int8 -> [-31, -61, -91, -121], int32 -> -2019179551
[OK  ] Квантизация: Triton == torch-эталон (бит-в-бит), 5 форм x 2 pack x 2 режима scale: 20/20
        например: 1x1
[OK  ] Квантизация: ошибка и нули (1024x1024), pack=8, per_row=False: 1/1
        rel_fro=0.2191 (теория 0.2224), max|err|=0.0380 <= scale=0.0760
[OK  ] Квантизация: ошибка и нули (1024x1024), pack=8, per_row=True: 1/1
        rel_fro=0.1422 (теория 0.1442), max|err|=0.0364 <= scale=0.0729
[OK  ] Квантизация: ошибка и нули (1024x1024), pack=32, per_row=False: 1/1
        rel_fro=0.2124 (теория 0.2154), max|err|=0.0368 <= scale=0.0736
[OK  ] Квантизация: ошибка и нули (1024x1024), pack=32, per_row=True: 1/1
        rel_fro=0.1416 (теория 0.1437), max|err|=0.0361 <= scale=0.0724
[OK  ] Память: ровно 4x, pack=8: 1/1
        2097152 Б (fp16) -> 524288 Б (int4)
[OK  ] Память: ров

## Бенчмарк ядер (общий)

In [4]:
from benchmark import run_benchmarks
run_benchmarks()

GPU: Tesla T4
== Квантизация fp16 -> int4, W (4096, 4096) ==
 pack     impl        ms      GB/s  speedup
    8   triton    1.0933      38.4     9.7x
    8    torch   10.5895       4.0     1.0x
   32   triton    0.6574      63.8    12.7x
   32    torch    8.3222       5.0     1.0x
(время Triton включает вычисление scale: abs().amax() в torch)

== Матмул bf16 x int4 -> bf16, W (4096, 4096) ==
 pack     M             impl        ms      GB/s  vs naive
    8     1      triton int4    1.5325       5.5      5.4x
    8     1      torch naive    8.3390       1.0      1.0x
    8     1    cuBLAS bf16 W    0.1999     168.0         -
    8    16      triton int4    1.5146       5.7      5.9x
    8    16      torch naive    8.9396       1.0      1.0x
    8    16    cuBLAS bf16 W    0.9081      37.2         -
    8  2048      triton int4   47.4712       0.9      0.9x
    8  2048      torch naive   44.0560       1.0      1.0x
    8  2048    cuBLAS bf16 W   38.5750       1.7         -
   32     1     

## X16@W4ᵀ vs X16@W16ᵀ на размерах Llama-3.2-1B

M ∈ {128, 512, 2048}, формы весов как у `unsloth/Llama-3.2-1B-Instruct`.

In [5]:
from bench_llama import run_bench
rows = run_bench(include_lm_head=True)

GPU: Tesla T4
Модель (размеры весов): unsloth/Llama-3.2-1B-Instruct
X, W16: bfloat16 | W4: int4 pack=8, per-row scale
M (токены): (128, 512, 2048)

layer            M      N      K     W4 ms    W16 ms   W4/W16   mem W4/W16
--------------------------------------------------------------------------------
q_proj         128   2048   2048    0.9948    0.4033    0.41x      25.00%
q_proj         512   2048   2048    3.0615    2.1381    0.70x      25.00%
q_proj        2048   2048   2048   12.0945   10.5329    0.87x      25.00%

k_proj         128    512   2048    0.7274    0.2936    0.40x      25.00%
k_proj         512    512   2048    1.1356    0.4120    0.36x      25.00%
k_proj        2048    512   2048    3.5792    1.9356    0.54x      25.00%

v_proj         128    512   2048    0.6047    0.2589    0.43x      25.00%
v_proj         512    512   2048    1.0851    0.4533    0.42x      25.00%
v_proj        2048    512   2048    3.5653    1.9975    0.56x      25.00%

o_proj         128   2048  

## QuantizedLinear на Llama-3.2-1B-Instruct

Загрузка модели (~2.5 ГБ bf16) → замена `nn.Linear` на `QuantizedLinear` → dummy forward.

In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda"

tokenizer = AutoTokenizer.from_pretrained("unsloth/Llama-3.2-1B-Instruct")
model = AutoModelForCausalLM.from_pretrained(
    "unsloth/Llama-3.2-1B-Instruct",
    dtype=torch.bfloat16,
    device_map=device,
)
model.eval()

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048, padding_idx=128004)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,)

In [7]:
import time

vocab = model.config.vocab_size

input_ids = torch.randint(0, min(vocab, 1000), (1, 16), device=device)
attention_mask = torch.ones_like(input_ids)

t0 = time.perf_counter()
out = model(input_ids=input_ids, attention_mask=attention_mask)
if torch.cuda.is_available():
    torch.cuda.synchronize()
dt = time.perf_counter() - t0
logits = out.logits

print(
    f"dummy forward: input_ids {tuple(input_ids.shape)} → logits {tuple(logits.shape)} "
    f"dtype={logits.dtype}  за {dt:.3f} с"
)

assert torch.isfinite(logits.float()).all(), "logits содержат NaN/Inf"

print("logits[0, -1, :5] =", logits[0, -1, :5].tolist())

dummy forward: input_ids (1, 16) → logits (1, 16, 128256) dtype=torch.bfloat16  за 0.240 с
logits[0, -1, :5] = [5.5625, 5.25, 6.03125, 5.0, 4.90625]


In [8]:
from quantize_llama import load_quantized_llama
from linear import QuantizedLinear

model, tokenizer = load_quantized_llama(
    pack_bits=8,
    per_row=True,
    quantize_lm_head=True,
)

q_proj = model.model.layers[0].self_attn.q_proj
assert isinstance(q_proj, QuantizedLinear), type(q_proj)
print("q_proj[0]:", q_proj)

Загрузка unsloth/Llama-3.2-1B-Instruct (torch.bfloat16)…


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

  Linear до квантизации: 113, память ≈ 2.47 ГБ
Квантизация Linear → QuantizedLinear (pack=8, per_row=True, skip=set())…
  заменено слоёв: 113
  осталось nn.Linear: 0, QuantizedLinear: 113
  память ≈ 1.15 ГБ  (×0.46 от исходной)
q_proj[0]: QuantizedLinear(in_features=2048, out_features=2048, pack_bits=8, bias=False)


In [9]:
vocab = model.config.vocab_size

input_ids = torch.randint(0, min(vocab, 1000), (1, 16), device=device)
attention_mask = torch.ones_like(input_ids)

t0 = time.perf_counter()
out = model(input_ids=input_ids, attention_mask=attention_mask)
if torch.cuda.is_available():
    torch.cuda.synchronize()
dt = time.perf_counter() - t0
logits = out.logits

print(
    f"dummy forward: input_ids {tuple(input_ids.shape)} → logits {tuple(logits.shape)} "
    f"dtype={logits.dtype}  за {dt:.3f} с"
)

assert torch.isfinite(logits.float()).all(), "logits содержат NaN/Inf"

print("logits[0, -1, :5] =", logits[0, -1, :5].tolist())

dummy forward: input_ids (1, 16) → logits (1, 16, 128256) dtype=torch.bfloat16  за 0.139 с
logits[0, -1, :5] = [3.75, 2.21875, 4.59375, 2.703125, 2.921875]
